# Notebook 05: Thực thi Toàn bộ End-to-End RAG Pipeline

Notebook này minh họa và kiểm thử toàn bộ luồng hoạt động của hệ thống Hỏi - Đáp RAG trên tài liệu tiếng Việt:
1. **Thiết lập & Nạp cấu hình hệ thống**
2. **Khởi tạo RAG Pipeline** (kết hợp `VectorRetriever`, `BGEReranker`, và `GroqLLM` / `OllamaLLM`)
3. **Hỏi đáp RAG chuẩn (Standard QA)** kèm trích dẫn nguồn tài liệu
4. **Kiểm thử trên cả 2 tài liệu thực tế** (`paper_Phân tích và xử lí ảnh.docx` & `paper_data mining.pdf`)
5. **Chế độ Streaming (Real-time Token Generation)**
6. **So sánh chất lượng: Có Reranker vs Không có Reranker**
7. **Kiểm thử Chống Ảo Giác (Anti-Hallucination Test)** với các câu hỏi ngoài phạm vi tài liệu
8. **Phân tích chi tiết độ trễ (Latency Breakdown)**

## 1. Thiết lập Môi trường & Imports

In [1]:
import os
import sys
import time
from pathlib import Path
from IPython.display import Markdown, display

# Xác định thư mục gốc của dự án (vietnamese-rag)
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import get_config
from src.logger import get_logger
from src.pipeline.rag_pipeline import RAGPipeline
from src.schemas.response import RAGResponse

print(f"✅ Project Root: {root}")

✅ Project Root: E:\Personal Project\vietnamese-rag


## 2. Kiểm tra Cấu hình Hệ thống (System Configuration)

In [2]:
%load_ext autoreload
%autoreload 2
cfg = get_config(reload=True)

print("=" * 65)
print(f" Ứng dụng        : {cfg.app.name} (v{cfg.app.version})")
print(f" Framework       : {cfg.app.framework}")
print(f" Vector Database : Qdrant ({cfg.vector_store.url})")
print(f" Collection      : {cfg.vector_store.collection_name}")
print(f" Embedding Model : {cfg.embedding.model_name} (dim: {cfg.embedding.dimension})")
print(f" Reranker        : {cfg.retrieval.reranker.provider} (enabled: {cfg.retrieval.reranker.enabled})")
print(f" LLM Provider    : {cfg.llm.default_provider} (model: {cfg.llm.groq.model_name})")
print("=" * 65)

 Ứng dụng        : vietnamese-rag (v0.1.0)
 Framework       : llamaindex
 Vector Database : Qdrant (http://localhost:6333)
 Collection      : vietnamese_docs_stage5_validation_20261017
 Embedding Model : BAAI/bge-m3 (dim: 1024)
 Reranker        : bge (enabled: True)
 LLM Provider    : groq (model: openai/gpt-oss-120b)


## 3. Khởi tạo RAG Pipeline Hoàn chỉnh

In [3]:
# Khởi tạo End-to-End Pipeline
rag = RAGPipeline()
print("✅ Khởi tạo RAG Pipeline thành công!")

2026-10-20 20:17:34 | INFO    | embedder:__init__:39 - Đang tải Embedding model: BAAI/bge-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-10-20 20:17:45 | INFO    | embedder:__init__:49 - Tải model BAAI/bge-m3 thành công. Kích thước vector: 1024
2026-10-20 20:17:45 | INFO    | reranker:__init__:37 - Đang tải Reranker model: BAAI/bge-reranker-v2-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

2026-10-20 20:17:52 | INFO    | reranker:__init__:45 - Tải Reranker model BAAI/bge-reranker-v2-m3 thành công!
2026-10-20 20:17:52 | INFO    | groq_llm:__init__:35 - Khởi tạo Groq LLM với mô hình: openai/gpt-oss-120b
2026-10-20 20:17:53 | INFO    | prompt_builder:_load_template:77 - Đã nạp prompt template từ: qa_prompt.yaml
✅ Khởi tạo RAG Pipeline thành công!


## 4. Hỏi đáp RAG Chuẩn (Standard QA) kèm Trích dẫn Nguồn

Thử nghiệm với câu hỏi từ tài liệu `paper_Phân tích và xử lí ảnh.docx`.

In [4]:
def display_rag_response(response: RAGResponse):
    """Hàm tiện ích hiển thị câu trả lời và trích dẫn trực quan trong Notebook."""
    display(Markdown(f"🤖 Câu trả lời của AI ({response.latency_seconds:.2f}s):\n\n" + response.answer))
    # print("-" * 80)
    # print(f"📚 NGUỒN TÀI LIỆU TRÍCH DẪN ({len(response.sources)} đoạn):")
    # for c in response.sources:
    #     meta = c.metadata or {}
    #     file_name = meta.get("file_name", "N/A")
    #     page = meta.get("page_number")
    #     section = meta.get("section_title")
    #     loc = f"Trang {page}" if page else (f"Mục: '{section}'" if section else "")
    #     print(f"  • [Hạng #{c.rank}] {file_name} ({loc}) | Điểm phù hợp: {c.score:.4f}")
    # print("=" * 80)

In [5]:
query_1 = "Mô hình đạt kết quả tốt nhất trong bài toán phân loại bệnh về răng miệng là gì và độ chính xác là bao nhiêu?"
query_3 = "Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?"
print(f"❓ Câu hỏi: {query_1}\n")
response_1 = rag.query(query_text=query_3, top_k=10, top_n=5)
display_rag_response(response_1)

❓ Câu hỏi: Mô hình đạt kết quả tốt nhất trong bài toán phân loại bệnh về răng miệng là gì và độ chính xác là bao nhiêu?

2026-10-20 20:18:19 | INFO    | rag_pipeline:query:53 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?'
2026-10-20 20:18:19 | INFO    | retrieval_pipeline:retrieve:125 - Retrieval Pipeline | hybrid=False | rerank=True | candidate_k=10 | final_n=5
2026-10-20 20:18:19 | INFO    | vector_retriever:retrieve:65 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_k=10)


The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


2026-10-20 20:18:21 | INFO    | vector_retriever:retrieve:119 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-20 20:18:21 | INFO    | retrieval_pipeline:retrieve:156 - Candidate Retrieval tra ve 10 chunks.
2026-10-20 20:18:21 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_n=5)
2026-10-20 20:18:31 | INFO    | reranker:rerank:85 - Hoàn tất Reranking. Lọc lại top 5 chunks chất lượng nhất.
2026-10-20 20:18:31 | INFO    | retrieval_pipeline:retrieve:203 - Pipeline hoan tat: 5 chunks cuoi cung.
2026-10-20 20:18:31 | INFO    | generator:_prepare_chunks:288 - Chunk preparation: input=5, output=5, duplicates_removed=0, empty_removed=0
2026-10-20 20:18:31 | INFO    | generator:generate_answer:314 - Đang gửi prompt đến LLM để trả lời query: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?'
2026-10-20 20:18:34 | INFO    | rag_pipeline:query:94 - Hoàn tất xử lý RAG Pipel

🤖 Câu trả lời của AI (14.94s):

Theo bảng **“Bảng 4: Quá trình huấn luyện mô hình DenseNet201”** trong tài liệu *paper_Phân tích và xử lí ảnh.docx* (đường dẫn: THỰC HÀNH > THỰC NGHIỆM VÀ ĐÁNH GIÁ > Thiết lập thực nghiệm > Kết quả huấn luyện và so sánh), tại **epoch 35** giá trị **Accuracy trên tập train** được ghi là **0.9597**, tương đương **95.97 %**.

- Nguồn: *paper_Phân tích và xử lí ảnh.docx*, Table 4, dòng “Epoch 35” (Acc = 0.9597)【Đoạn ngữ cảnh 4】.  
- Ngoài ra, trong phần mô tả tổng quan cũng nhắc lại rằng “accuracy trên tập huấn luyện là 95.97%” tại epoch 35【Đoạn ngữ cảnh 1】.

## 5. Thử nghiệm Câu hỏi từ Tài liệu Thứ Hai (`paper_data mining.pdf`)

Kiểm tra khả năng truy xuất trên bài báo khoa học về Khai phá dữ liệu hành vi khách hàng.

In [ ]:
query_2 = "Đề tài khai phá dữ liệu hành vi khách hàng giải quyết bài toán gì trong lĩnh vực thể hình và thương mại điện tử?"

print(f"❓ Câu hỏi: {query_2}\n")
response_2 = rag.query(query_text=query_2, top_k=10, top_n=3)
display_rag_response(response_2)

❓ Câu hỏi: Đề tài khai phá dữ liệu hành vi khách hàng giải quyết bài toán gì trong lĩnh vực thể hình và thương mại điện tử?

2026-10-06 09:45:28 | INFO    | rag_pipeline:query:40 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Đề tài khai phá dữ liệu hành vi khách hàng giải quyết bài toán gì trong lĩnh vực thể hình và thương mại điện tử?'
2026-10-06 09:45:28 | INFO    | retrieval_pipeline:retrieve:40 - Bắt đầu chu trình Retrieval Pipeline cho query: 'Đề tài khai phá dữ liệu hành vi khách hàng giải quyết bài toán gì trong lĩnh vực thể hình và thương mại điện tử?'
2026-10-06 09:45:28 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Đề tài khai phá dữ liệu hành vi khách hàng giải quyết bài toán gì trong lĩnh vực thể hình và thương mại điện tử?' (top_k=10)
2026-10-06 09:45:28 | INFO    | vector_retriever:retrieve:82 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-06 09:45:28 | INFO    | retrieval_pipeline:retrieve:56 - Reranker không được kích hoạt, sử 

🤖 Câu trả lời của AI (2.95s):

**Đề tài khai phá dữ liệu hành vi khách hàng** hướng tới việc giải quyết các bài toán sau trong hai lĩnh vực **thể hình** và **thương mại điện tử**:

| Lĩnh vực | Bài toán được giải quyết | Nội dung chi tiết |
|----------|--------------------------|-------------------|
| **Thể hình** | Nhận diện đặc điểm hội viên để **quản lý và duy trì sự gắn kết** của khách hàng. | - Phân tích dữ liệu hội viên (Fitness Membership) nhằm xác định các nhóm khách hàng trung thành và các nhóm có nguy cơ rời bỏ.<br>- Từ đó đề xuất các chiến lược giữ chân, cá nhân hoá dịch vụ cho từng nhóm.<br>*(Nguồn: Tài liệu tham khảo 2, trang 12)* |
| **Thương mại điện tử** | Phân đoạn khách hàng **chính xác** để thực hiện **tiếp thị cá nhân hoá**, tăng **chuyển đổi** và **tối ưu doanh thu** bền vững. | - Khai thác dữ liệu mua sắm (Customer Shopping Behavior) để xác định các hành vi, sở thích và nhu cầu mua hàng.<br>- Xây dựng các segment khách hàng, từ đó thiết kế các chiến dịch marketing nhắm mục tiêu, cải thiện tỷ lệ chuyển đổi và doanh thu.<br>*(Nguồn: Tài liệu tham khảo 2, trang 12)* |
| **Cả hai lĩnh vực** | **Xử lý, làm sạch dữ liệu** và **phân tích khám phá dữ liệu (EDA)** để trích xuất insight về hành vi người dùng. | - Thực hiện các bước chuẩn bị dữ liệu (Data Cleaning & Preprocessing).<br>- Thực hiện EDA để khám phá các mẫu, xu hướng và mối quan hệ trong dữ liệu, từ đó rút ra các đặc điểm chung của các nhóm khách hàng.<br>*(Nguồn: Tài liệu tham khảo 3, trang 12)* |
| **Kết quả tổng thể** | Cung cấp cái nhìn toàn diện về **ứng dụng học máy trong phân tích hành vi người dùng**, đáp ứng các mục tiêu đề ra trong chương 1 của đề tài. | - Đánh giá chi tiết trên hai bộ dữ liệu thực tế (Fitness Membership Analytics và Customer Shopping Behavior).<br>*(Nguồn: Tài liệu tham khảo 1, trang 69)* |

**Tóm lại**, đề tài khai phá dữ liệu hành vi khách hàng giải quyết:  
1. **Trong lĩnh vực thể hình** – việc nhận diện và quản lý đặc điểm hội viên để duy trì gắn kết và tăng độ trung thành.  
2. **Trong lĩnh vực thương mại điện tử** – việc phân đoạn khách hàng để thực hiện tiếp thị cá nhân hoá, nâng cao tỷ lệ chuyển đổi và tối ưu doanh thu.  

Các giải pháp này được thực hiện thông qua quy trình xử lý dữ liệu, khám phá dữ liệu và trích xuất insight từ hai bộ dữ liệu thực tế.  

*Tham khảo:*  
- Tài liệu tham khảo 2, trang 12.  
- Tài liệu tham khảo 3, trang 12.  
- Tài liệu tham khảo 1, trang 69.

--------------------------------------------------------------------------------
📚 NGUỒN TÀI LIỆU TRÍCH DẪN (3 đoạn):
  • [Hạng #1] paper_data mining.pdf (Trang 69) | Điểm phù hợp: 0.6175
  • [Hạng #2] paper_data mining.pdf (Trang 12) | Điểm phù hợp: 0.6136
  • [Hạng #3] paper_data mining.pdf (Trang 12) | Điểm phù hợp: 0.6045


## 6. Thử nghiệm Chế độ Streaming (Real-time Token Generation)

Chế độ này hiển thị từng token thời gian thực khi mô hình LLM đang sinh văn bản.

In [7]:
stream_query = "Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?"

print(f"❓ Câu hỏi: {stream_query}\n")
chunks, stream_iter = rag.stream_query(query=stream_query, top_k=10, top_n=5)

print(f" Đã tìm thấy {len(chunks)} đoạn ngữ cảnh liên quan.")
print("🤖 AI đang bắt đầu sinh câu trả lời:\n")

for token in stream_iter:
    print(token, end="", flush=True)

print("\n\n" + "=" * 80)

❓ Câu hỏi: Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?

2026-10-20 20:27:32 | INFO    | rag_pipeline:stream_query:128 - Bắt đầu Stream RAG Pipeline cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?'
2026-10-20 20:27:32 | INFO    | retrieval_pipeline:retrieve:125 - Retrieval Pipeline | hybrid=False | rerank=True | candidate_k=10 | final_n=5
2026-10-20 20:27:32 | INFO    | vector_retriever:retrieve:65 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_k=10)
2026-10-20 20:27:32 | INFO    | vector_retriever:retrieve:119 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
2026-10-20 20:27:32 | INFO    | retrieval_pipeline:retrieve:156 - Candidate Retrieval tra ve 10 chunks.
2026-10-20 20:27:32 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_n=5)
2026-10-20 20:27:38 | INFO    | rer

## 7. So sánh: Có Reranker vs Không có Reranker

So sánh thứ hạng điểm số giữa Vector Search đơn thuần (Bi-Encoder) và Semantic Cross-Encoder Reranker.

In [ ]:
compare_query = "Kiến trúc mạng DenseNet201 gồm những tầng và tham số nào?"

print("=" * 75)
print(" 1. KẾT QUẢ TỪ VECTOR SEARCH BAN ĐẦU (Bi-Encoder, top 5):")
initial_chunks = rag.retrieval_pipeline.retriever.retrieve(compare_query, top_k=5)
for idx, c in enumerate(initial_chunks, start=1):
    sec = c.metadata.get("section_title", "N/A")
    print(f"  #{idx} [Score: {c.score:.4f}] - Mục: {sec}")
    print(f"      Nội dung: {c.text[:130]}...\n")

if rag.retrieval_pipeline.reranker is not None:
    print("=" * 75)
    print("🔹 2. KẾT QUẢ SAU KHI QUA SEMANTIC RERANKER (Cross-Encoder, top 3):")
    reranked_chunks = rag.retrieval_pipeline.reranker.rerank(compare_query, initial_chunks, top_n=3)
    for idx, c in enumerate(reranked_chunks, start=1):
        sec = c.metadata.get("section_title", "N/A")
        print(f"  #{idx} [Score: {c.score:.4f}] - Mục: {sec}")
        print(f"      Nội dung: {c.text[:130]}...\n")
else:
    print(" Reranker hiện đang tắt trong configs/config.yaml.")

 1. KẾT QUẢ TỪ VECTOR SEARCH BAN ĐẦU (Bi-Encoder, top 5):
2026-10-06 10:15:34 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Kiến trúc mạng DenseNet201 gồm những tầng và tham số nào?' (top_k=5)


The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


2026-10-06 10:15:35 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 5 chunks tương đồng từ Qdrant.
  #1 [Score: 0.6837] - Mục: Sơ đồ kiến trúc mạng DenseNet201
      Nội dung: Cơ chế hoạt động của mô hình:

Mô hình DenseNet-201 nhận đầu vào là ảnh X-quang kích thước 224×224×3 (chiều cao, chiều rộng, 3 kên...

  #2 [Score: 0.6734] - Mục: Đánh giá trên tập test
      Nội dung: Trong khi đó, mô hình EfficientNetB0 đạt điểm tối ưu tại Epoch thứ 26 với độ chính xác trên tập kiểm thử là 91% và giá trị Validat...

  #3 [Score: 0.6344] - Mục: Đánh giá trên tập test
      Nội dung: Mặc dù cả hai mô hình đều hoạt động hiệu quả và đáp ứng tốt yêu cầu thực tiễn của bài toán, nhưng dựa trên sự ổn định và hiệu suất...

  #4 [Score: 0.6294] - Mục: Tham số của mô hình
      Nội dung: ## Tham số của mô hình

Các tham số chính của mô hình được thiết lập như sau:

Bảng 2: Bảng mô tả tham số chính của mô hình DenseN...

  #5 [Score: 0.6285] - Mục: Kiến trúc mạng Densenet201
      Nội dung: ## Kiến 

## 8. Kiểm thử Khả năng Chống Ảo Giác (Anti-Hallucination)

Thử thách mô hình bằng một câu hỏi nằm ngoài phạm vi tài liệu để xem AI có tự bịa thông tin hay trung thực từ chối.

In [7]:
out_of_domain_query = "Thủ đô của nước Pháp là gì và dân số hiện tại là bao nhiêu?"

print(f"❓ Câu hỏi ngoài lề: {out_of_domain_query}\n")
response_hal = rag.query(query_text=out_of_domain_query, top_k=5, top_n=3)
display_rag_response(response_hal)

❓ Câu hỏi ngoài lề: Thủ đô của nước Pháp là gì và dân số hiện tại là bao nhiêu?

2026-10-06 10:24:37 | INFO    | rag_pipeline:query:46 - Bắt đầu chu trình RAG Pipeline cho câu hỏi: 'Thủ đô của nước Pháp là gì và dân số hiện tại là bao nhiêu?'
2026-10-06 10:24:37 | INFO    | retrieval_pipeline:retrieve:59 - Retrieval Pipeline cho query: 'Thủ đô của nước Pháp là gì và dân số hiện tại là bao nhiêu?' (hybrid=False, rerank=True, top_k=5, top_n=3)
2026-10-06 10:24:37 | INFO    | vector_retriever:retrieve:49 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Thủ đô của nước Pháp là gì và dân số hiện tại là bao nhiêu?' (top_k=5)
2026-10-06 10:24:37 | INFO    | vector_retriever:retrieve:90 - Đã tìm thấy 5 chunks tương đồng từ Qdrant.
2026-10-06 10:24:37 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 5 chunks với câu hỏi: 'Thủ đô của nước Pháp là gì và dân số hiện tại là bao nhiêu?' (top_n=3)
2026-10-06 10:24:39 | INFO    | reranker:rerank:85 - Hoàn tất Reranking. Lọc lại top 3 chunks chất lượng nh

🤖 Câu trả lời của AI (2.76s):

Dựa trên tài liệu được cung cấp, tôi không tìm thấy thông tin đầy đủ để trả lời câu hỏi này.

--------------------------------------------------------------------------------
📚 NGUỒN TÀI LIỆU TRÍCH DẪN (3 đoạn):
  • [Hạng #1] paper_data mining.pdf (Trang 26) | Điểm phù hợp: 0.0022
  • [Hạng #2] paper_data mining.pdf (Trang 27) | Điểm phù hợp: 0.0011
  • [Hạng #3] paper_data mining.pdf (Trang 26) | Điểm phù hợp: 0.0006


## 9. Đo lường Chi tiết Độ Trễ (Latency Breakdown)

Phân tích tỷ trọng thời gian tiêu tốn cho từng công đoạn: Embedding $\rightarrow$ Qdrant Search $\rightarrow$ Reranker $\rightarrow$ LLM.

In [ ]:
benchmark_query = "Mục tiêu nghiên cứu phân đoạn ảnh X-quang"

# 1. Thời gian tạo embedding cho query
t0 = time.time()
query_vector = rag.retrieval_pipeline.retriever.embedder.embed_query(benchmark_query)
t_embed = time.time() - t0

# 2. Thời gian tìm kiếm trên Qdrant
t0 = time.time()
candidate_chunks = rag.retrieval_pipeline.retriever.retrieve(benchmark_query, top_k=10)
t_qdrant = time.time() - t0

# 3. Thời gian Reranking (nếu có)
if rag.retrieval_pipeline.reranker is not None:
    t0 = time.time()
    top_chunks = rag.retrieval_pipeline.reranker.rerank(benchmark_query, candidate_chunks, top_n=3)
    t_rerank = time.time() - t0
else:
    top_chunks = candidate_chunks[:3]
    t_rerank = 0.0

# 4. Thời gian sinh câu trả lời LLM
t0 = time.time()
answer = rag.generator.generate_answer(benchmark_query, top_chunks)
t_llm = time.time() - t0

total_time = t_embed + t_qdrant + t_rerank + t_llm

print("=" * 60)
print("⏱️  PHÂN TÍCH ĐỘ TRỄ CHI TIẾT (LATENCY BREAKDOWN):")
print(f"  • 1. Query Embedding   : {t_embed:.3f}s ({t_embed/total_time*100:>5.1f}%)")
print(f"  • 2. Qdrant Search     : {t_qdrant:.3f}s ({t_qdrant/total_time*100:>5.1f}%)")
print(f"  • 3. Semantic Rerank   : {t_rerank:.3f}s ({t_rerank/total_time*100:>5.1f}%)")
print(f"  • 4. LLM Generation    : {t_llm:.3f}s ({t_llm/total_time*100:>5.1f}%)")
print(f" -----------------------------------------------")
print(f"  👉 TỔNG THỜI GIAN      : {total_time:.3f}s (100.0%)")
print("=" * 60)